<a href="https://colab.research.google.com/github/broadinstitute/BE3D/blob/main/examples/BE3Dv6_MultipleScreensComplex_LocalScriptNotebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# README
- This is an example for MEN1 which is based on two CBE screens


# Setup

In [ ]:
# @title Install DSSP and ClustalO

! apt-get update
! apt-get install dssp clustalo


Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
clustalo is already the newest version (1.2.4-7).
dssp is already the newest version (4.0.4-1)

In [ ]:
# @title Connect to Google Drive
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# @title Install BE3D

! pip install git+https://github.com/broadinstitute/beclust3d-public.git
print('beclust3d installed at:')
! pip show beclust3d


  Cloning https://github.com/broadinstitute/beclust3d-public.git to /tmp/pip-req-build-kbuxi562
  Running command git clone --filter=blob:none --quiet https://github.com/broadinstitute/beclust3d-public.git /tmp/pip-req-build-kbuxi562
  Resolved https://github.com/broadinstitute/beclust3d-public.git to commit af929f3b3d2c7e6e8b9f61d01fad0002626288e0
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
beclust3d installed at:
Name: beclust3d
Version: 1.0.0
Summary: Package for prioritizing and clustering base editing screen scores based on structure and conservation.
Home-page: 
Author: 
Author-email: Calvin XiaoYang Hu <xiaohu@g.harvard.edu>, Yoochan Myung <ymyung@broadinstitute.org>
License: Copyright (c) 2025 Sumaiya Iqbal, Calvin XiaoYang Hu, Yoochan Myung

Permission is hereby granted, free of charge, to any person obtaining a copy of this software and associated documentation files (the "Software"), t

In [ ]:
# @title Download relevant files

! wget https://raw.githubusercontent.com/broadinstitute/BE3D/refs/heads/main/examples/pdb/KBTBD4_8voj_single_chain_A.pdb -O KBTBD4_8voj_single_chain_A.pdb
! wget https://raw.githubusercontent.com/broadinstitute/BE3D/refs/heads/main/examples/data/YeoNature2025-ABE-Screen-NegCtrl.tsv -O YeoNature2025-ABE-Screen-NegCtrl.tsv
! wget https://raw.githubusercontent.com/broadinstitute/BE3D/refs/heads/main/examples/data/YeoNature2025-CBE-Screen-NegCtrl.tsv -O YeoNature2025-CBE-Screen-NegCtrl.tsv

! mkdir temp
! wget https://raw.githubusercontent.com/broadinstitute/BE3D/refs/heads/main/examples/yaml/KBTBD4-ChainA.yaml -O temp/KBTBD4-ChainA.yaml

! wget https://raw.githubusercontent.com/broadinstitute/BE3D/refs/heads/main/examples/beclust3d_local.py -O beclust3d_local.py


--2026-02-18 04:22:05--  https://raw.githubusercontent.com/broadinstitute/BE3D/refs/heads/main/examples/pdb/KBTBD4_8voj_single_chain_A.pdb
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.110.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 320355 (313K) [text/plain]
Saving to: ‘KBTBD4_8voj_single_chain_A.pdb’

KBTBD4_8voj_single_ 100%[===================>] 312.85K  --.-KB/s    in 0.006s  

2026-02-18 04:22:05 (51.9 MB/s) - ‘KBTBD4_8voj_single_chain_A.pdb’ saved [320355/320355]

--2026-02-18 04:22:05--  https://raw.githubusercontent.com/broadinstitute/BE3D/refs/heads/main/examples/data/YeoNature2025-ABE-Screen-NegCtrl.tsv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|18

In [ ]:
# @title Import packages

import os
import sys
import yaml
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import gc
import uuid
import ipywidgets as widgets
from IPython.display import display

from IPython.display import Image, display, SVG
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from google.colab import files
import shutil

from beclust3d import *

import logging
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)


# Input (optional)
This section creates a .yaml file, so this step is not necessary if you use the .yaml file from the Github as is

In [ ]:
# @title Required (filled with default values for KBTBD4)

yaml_dict = {}

beclust3d_path = "../"  # @param {type:"string"}
yaml_dict['beclust3d_path'] = beclust3d_path

screen_dir = "/content/"  # @param {type:"string"}
yaml_dict['screen_dir'] = screen_dir

nRandom = 500  # @param {type:"integer"}
yaml_dict['nRandom'] = nRandom

# ---- pthr ----
yaml_dict['pthr'] = {}

single_screen = 0.05  # @param {type:"number"}
yaml_dict['pthr']['single_screen'] = single_screen

multi_screen = 0.05  # @param {type:"number"}
yaml_dict['pthr']['multi_screen'] = multi_screen

structure_radius = 6.0  # @param {type:"number"}
yaml_dict['structure_radius'] = structure_radius

clustering_radius = 6.0  # @param {type:"number"}
yaml_dict['clustering_radius'] = clustering_radius

function_for_lfc = "max"  # @param {type:"string"}
yaml_dict['function_for_lfc'] = function_for_lfc

function_for_lfc3d = "mean"  # @param {type:"string"}
yaml_dict['function_for_lfc3d'] = function_for_lfc3d

function_for_meta = "SUM"  # @param {type:"string"}
yaml_dict['function_for_meta'] = function_for_meta

# ---- database ----
yaml_dict['database'] = {}

mut_list_col = "Mutation category"  # @param {type:"string"}
yaml_dict['database']['mut_list_col'] = mut_list_col

mut_col = "Mutation_type"  # @param {type:"string"}
yaml_dict['database']['mut_col'] = mut_col

val_col = "lfc_over_unsorted"  # @param {type:"string"}
yaml_dict['database']['val_col'] = val_col

gene_col = "Gene Symbol"  # @param {type:"string"}
yaml_dict['database']['gene_col'] = gene_col

edits_col = "Amino acid edits"  # @param {type:"string"}
yaml_dict['database']['edits_col'] = edits_col

mut_delimiter = ";"  # @param {type:"string"}
yaml_dict['database']['mut_delimiter'] = mut_delimiter

gRNA_col = None  # @param {type:"string"}
yaml_dict['database']['gRNA_col'] = gRNA_col

# ---- conservation ----
yaml_dict['conservation'] = {}

conservation_run = False  # @param {type:"boolean"}
yaml_dict['conservation']['run'] = conservation_run

v_score_threshold = 3  # @param {type:"integer"}
yaml_dict['conservation']['v_score_threshold'] = v_score_threshold

alt_gene_name = None  # @param {type:"string"}
yaml_dict['conservation']['alt_gene_name'] = alt_gene_name

alt_uniprot_id = None  # @param {type:"string"}
yaml_dict['conservation']['alt_uniprot_id'] = alt_uniprot_id

alt_screen_start = None  # @param {type:"string"}
yaml_dict['conservation']['alt_screen_start'] = alt_screen_start

# ---- mutation_category ----
yaml_dict['mutation_category'] = {}

missense = ["Missense"]  # @param {type:"raw"}
yaml_dict['mutation_category']['missense'] = missense

silent = ["Silent"]  # @param {type:"raw"}
yaml_dict['mutation_category']['silent'] = silent

nonsense = ["Nonsense"]  # @param {type:"raw"}
yaml_dict['mutation_category']['nonsense'] = nonsense

no_mutation = ["No Mutation"]  # @param {type:"raw"}
yaml_dict['mutation_category']['no_mutation'] = no_mutation

splice = ["Splice-donor", "Splice-acceptor"]  # @param {type:"raw"}
yaml_dict['mutation_category']['splice'] = splice

intron = ["Intron"]  # @param {type:"raw"}
yaml_dict['mutation_category']['intron'] = intron

# ---- qa ----
yaml_dict['qa'] = {}

qa_passed_only = False  # @param {type:"boolean"}
yaml_dict['qa']['qa_passed_only'] = qa_passed_only

qa_only = False  # @param {type:"boolean"}
yaml_dict['qa']['qa_only'] = qa_only

cases = ["Nonsense", "Splice-donor", "Splice-acceptor"]  # @param {type:"raw"}
yaml_dict['qa']['cases'] = cases

controls = ["No Mutation"]  # @param {type:"raw"}
yaml_dict['qa']['controls'] = controls

input_gene = "KBTBD4"  # @param {type:"string"}
yaml_dict['input_gene'] = input_gene

input_uniprot = "Q9NVX7"  # @param {type:"string"}
yaml_dict['input_uniprot'] = input_uniprot

input_chain = "A"  # @param {type:"string"}
yaml_dict['input_chain'] = input_chain

screens = "YeoNature2025-ABE-Screen-NegCtrl.tsv,YeoNature2025-CBE-Screen-NegCtrl.tsv"  # @param {type:"string"}
yaml_dict['screens'] = screens

output_dir = ""  # @param {type:"string"}
yaml_dict['output_dir'] = output_dir

user_fasta = None  # @param {type:"string"}
yaml_dict['user_fasta'] = user_fasta

user_pdb = "KBTBD4_8voj_single_chain_A.pdb"  # @param {type:"string"}
yaml_dict['user_pdb'] = user_pdb

user_dssp = None  # @param {type:"string"}
yaml_dict['user_dssp'] = user_dssp

priority_on_alternative = False  # @param {type:"boolean"}
yaml_dict['priority_on_alternative'] = priority_on_alternative

ppi_chain_gene_dict = None  # @param {type:"raw"}
yaml_dict['ppi_chain_gene_dict'] = ppi_chain_gene_dict

ppi_gene_edits_dict = None  # @param {type:"raw"}
yaml_dict['ppi_gene_edits_dict'] = ppi_gene_edits_dict

atom_level_naa = False  # @param {type:"boolean"}
yaml_dict['atom_level_naa'] = atom_level_naa


In [ ]:
# @title Convert dictionary to yaml file

import yaml

yaml_filename = 'temp/KBTBD4-ChainA.yaml' # @param {type:"string"}

with open(yaml_filename, 'w') as file:
    yaml.dump(yaml_dict, file, sort_keys=False, default_flow_style=False)

print(f"YAML file '{yaml_filename}' created successfully.")


YAML file 'temp/KBTBD4-ChainA.yaml' created successfully.


# Running BE3D

In [ ]:
# Run script

! python beclust3d_local.py temp/KBTBD4-ChainA.yaml


All results will be saved in the following directory:
/content/
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not 

In [ ]:
# @title Download output directory
if output_dir == '/content/':
    print(f'Downloading {output_dir} is not recommended. Try downloading specific folders or changing [output_dir].')

download_directory = False #@param {type:"boolean"}
if download_directory:
    shutil.make_archive(output_dir, 'zip', output_dir)
    files.download(f"{output_dir}.zip")
